# 1. 과제 안내 + 주간 테스트 범위

In [1]:
# -*- coding: utf-8 -*-
import re


DEFAULT_NONE = "과제 없음"

def clean(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()


def parse_book(line: str):
    pre = mid = post = ""

    m = re.search(r"전\s*:\s*([^/]+)", line)
    if m:
        pre = clean(m.group(1))

    m = re.search(r"중\s*:\s*([^/]+)", line)
    if m:
        mid = clean(m.group(1))

    m = re.search(r"후\s*:\s*(.+)$", line)
    if m:
        post = clean(m.group(1))

    return pre, mid, post


def parse_pair(line: str, a_key: str, b_key: str):
    a = b = ""

    m = re.search(rf"{re.escape(a_key)}\s*:\s*([^⦁]+)", line)
    if m:
        a = clean(m.group(1))

    m = re.search(rf"{re.escape(b_key)}\s*:\s*(.+)$", line)
    if m:
        b = clean(m.group(1))

    return a, b


def split_by_weekly(text: str):
    lines = [clean(l) for l in text.replace("\r", "").split("\n") if clean(l)]
    idx = None
    for i, l in enumerate(lines):
        if "주간테스트" in l:
            idx = i
            break
    if idx is None:
        return lines, []
    return lines[:idx], lines[idx + 1:]


def tokenize_weekly(lines: list[str]) -> list[str]:
    combined = " ".join(lines).replace("▹", " ")
    return [clean(p) for p in combined.split("✔") if clean(p)]


def _parse_weekly_line(tok: str, label: str) -> str | None:
    """
    주간테스트 토큰에서
    - '문해 문학 - 실전, 과제 : 50~53쪽' 같은 형태도
    - '문해 문학 - 정기 평가에서 출제' (콜론 없는 형태)도
    처리한다.
    반환은 '왼쪽설명: 오른쪽값' 또는 '정기 평가에서 출제' 같은 문장.
    """
    tok = clean(tok)

    # 1) 콜론 있는 일반형
    m = re.search(rf"{re.escape(label)}\s*-\s*([^:]+)\s*:\s*(.+)$", tok)
    if m:
        return f"{clean(m.group(1))}: {clean(m.group(2))}"

    # 2) 콜론 없는 예외형 (예: '문해 문학 - 정기 평가에서 출제 : 3쪽' 또는 ': 없이 끝')
    m = re.search(rf"{re.escape(label)}\s*-\s*(.+)$", tok)
    if m:
        return clean(m.group(1))

    return None


def parse_weekly(tokens: list[str]) -> dict:
    res = {"book": "", "read": "", "lit": "", "vocab": "", "gram": ""}

    for tok in tokens:
        tok = clean(tok)

        if tok.startswith("도서 읽기") or tok.startswith("독서논술"):
            # 도서 읽기 - 독서 퀴즈, 요약하기 : 114~120쪽
            label = "독서논술" if tok.startswith("독서논술") else "도서 읽기"
            v = _parse_weekly_line(tok, label)
            if v:
                res["book"] = v

        elif tok.startswith("문해 독서"):
            v = _parse_weekly_line(tok, "문해 독서")
            if v:
                res["read"] = v

        elif tok.startswith("문해 문학"):
            v = _parse_weekly_line(tok, "문해 문학")
            if v:
                res["lit"] = v

        elif tok.startswith("어휘"):
            # 어휘 : 94~97, 110~111쪽
            m = re.search(r"어휘\s*:\s*(.+)$", tok)
            if m:
                res["vocab"] = clean(m.group(1))

        elif tok.startswith("문법"):
            v = _parse_weekly_line(tok, "문법")
            if v:
                res["gram"] = v

    return res


def parse_tasks(task_lines: list[str]) -> dict:
    t = {
        "book_pre": "", "book_mid": "", "book_post": "",
        "rd_jud": "", "rd_task": "",
        "lit_jud": "", "lit_task": "",
        "voc_study": "", "voc_quiz": "",
        "gram_task": "",
    }

    for raw in task_lines:
        line = clean(raw.replace("✔", ""))

        if ("도서 읽기" in line) or ("독서논술" in line):
            t["book_pre"], t["book_mid"], t["book_post"] = parse_book(line)

        elif "문해 독서" in line:
            t["rd_jud"], t["rd_task"] = parse_pair(line, "명제 판단", "과제 학습")

        elif "문해 문학" in line:
            t["lit_jud"], t["lit_task"] = parse_pair(line, "명제 판단", "과제 학습")

        elif "문해 어휘" in line:
            t["voc_study"], t["voc_quiz"] = parse_pair(line, "어휘 학습", "어휘 퀴즈")

        elif line.startswith("문법"):
            m = re.search(r"과제\s*학습\s*:\s*(.+)$", line)
            if m:
                t["gram_task"] = clean(m.group(1))

    return t


def apply_missing_defaults(tasks: dict) -> dict:
    """
    ✅ 예외 규칙
    - 입력에 문해 문학/문법이 아예 없을 수 있음
    - 그 경우 출력은 '과제 없음'으로 채운다.
    """
    def fill(key: str):
        if not clean(tasks.get(key, "")):
            tasks[key] = DEFAULT_NONE

    # 문해 문학(명제/과제)
    fill("lit_jud")
    fill("lit_task")
    # 문법(과제)
    fill("gram_task")

    return tasks


def render(tasks: dict, weekly: dict) -> str:
    return "\n".join([
        "1. 독서논술",
        f"- 전: {tasks['book_pre']} / 중 : {tasks['book_mid']} / 후 : {tasks['book_post']}",
        "",
        "2. 문해 독서",
        f"- 명제 판단: {tasks['rd_jud']}",
        f"- 과제 학습: {tasks['rd_task']}",
        "",
        "3. 문해 문학",
        f"- 명제 판단: {tasks['lit_jud']}",
        f"- 과제 학습: {tasks['lit_task']}",
        "",
        "4. 문해 어휘",
        f"- 어휘 학습: {tasks['voc_study']}",
        f"- 어휘 퀴즈: {tasks['voc_quiz']}",
        "",
        "5. 문법",
        f"- 과제 학습: {tasks['gram_task']}",
        "",
        "◇ 주간 테스트 범위 ◇",
        f"1. 독서논술 - {weekly['book']}",
        f"2. 문해 독서 - {weekly['read']}",
        f"3. 문해 문학 - {weekly['lit']}",
        f"4. 어휘: {weekly['vocab']}",
        f"5. 문법 - {weekly['gram']}",
    ])


def convert(text: str) -> str:
    task_lines, weekly_lines = split_by_weekly(text)
    tasks = parse_tasks(task_lines)
    tasks = apply_missing_defaults(tasks)  # ✅ 추가된 예외 처리
    weekly = parse_weekly(tokenize_weekly(weekly_lines))
    return render(tasks, weekly)


In [ ]:
raw_text = """ 
2주차 수업 시간( 월 일)까지 수행해 오기 (완료한 과제 앞에 표시하기)
✔ 독서논술 ⦁전 : 70~77쪽 / 중 : 78~90쪽 / 후 : 27쪽
✔ 문해 독서 ⦁명제 판단 : 34~39쪽(선택지 유형 고르기) ⦁과제 학습 : 40~43쪽
✔ 문해 문학 ⦁명제 판단 : 46~49쪽(선택지 유형 고르기) ⦁과제 학습 : 50~54쪽
✔ 문해 어휘 ⦁어휘 학습 : 56~59쪽 ⦁어휘 퀴즈 : 60~61쪽(어휘 공부 후 문제 풀기)
✔ 문법 ⦁과제 학습 : 65~67쪽(개념 복습 후 문제 풀기)
▹ 2주차 수업 시간에 볼 주간테스트 범위
✔ 독서논술 - 독서 퀴즈, 요약하기 : 82~90쪽 ✔ 문해 독서 - 실전, 과제 : 28~43쪽 ✔ 문해 문학 - 개념 2문항, 과제 : 50~54쪽
✔ 어휘 : 56~59, 78~79쪽 ✔ 문법 - 실전, 과제 : 62~67쪽
"""
print(convert(raw_text))

1. 독서논술
- 전: 70~77쪽 / 중 : 78~90쪽 / 후 : 27쪽

2. 문해 독서
- 명제 판단: 34~39쪽(선택지 유형 고르기)
- 과제 학습: 40~43쪽

3. 문해 문학
- 명제 판단: 46~49쪽(선택지 유형 고르기)
- 과제 학습: 50~54쪽

4. 문해 어휘
- 어휘 학습: 56~59쪽
- 어휘 퀴즈: 60~61쪽(어휘 공부 후 문제 풀기)

5. 문법
- 과제 학습: 65~67쪽(개념 복습 후 문제 풀기)

◇ 주간 테스트 범위 ◇
1. 독서논술 - 독서 퀴즈, 요약하기: 82~90쪽
2. 문해 독서 - 실전, 과제: 28~43쪽
3. 문해 문학 - 개념 2문항, 과제: 50~54쪽
4. 어휘: 56~59, 78~79쪽
5. 문법 - 실전, 과제: 62~67쪽


: 

In [ ]:
# %pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# -*- coding: utf-8 -*-
import re
from pathlib import Path
import pandas as pd

EXCEL_PATH = r"C:\Users\9191h\Desktop\[중1] - 1학기 진도 사항\main.xlsx"

# ✅ 저장 위치: 바탕화면 우선
DESKTOP = Path.home() / "Desktop\[중1] - 1학기 진도 사항\진도_내용"
OUTPUT_DIR = DESKTOP if DESKTOP.exists() else Path.cwd()

DEFAULT_NONE = "과제 없음"


# -------------------------
# 기본 유틸
# -------------------------
def clean(x) -> str:
    """None/NaN/특수공백 처리 + 양끝 공백 제거"""
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return str(x).replace("\u00a0", " ").strip()


def row_join(row) -> str:
    """행 전체를 합쳐서 문자열 탐색용으로 사용"""
    parts = []
    for v in row.tolist():
        t = clean(v)
        if t:
            parts.append(t)
    return " ".join(parts)


def week_sort_key(w: str) -> int:
    m = re.search(r"\d+", w)
    return int(m.group()) if m else 999


# -------------------------
# 감지/파싱 함수들
# -------------------------
def parse_issue_no(text: str) -> int | None:
    """
    '중1 1호', '중 1-1호', '중Ⅰ 1호' 등 최대한 허용
    """
    t = clean(text)
    m = re.search(r"중\s*[1ⅠI]\s*[-\s]*\[*\s*(\d+)\s*\]*\s*호", t)
    return int(m.group(1)) if m else None


def parse_week(text: str) -> tuple[str | None, bool]:
    """
    '2주차', '2주차(정기평가)' 등에서 주차+정기평가 여부 반환
    """
    t = clean(text)
    m = re.search(r"(\d+)\s*주차", t)
    if not m:
        return None, False
    week = f"{m.group(1)}주차"
    is_exam = "정기평가" in t
    return week, is_exam


def detect_area_from_text(text: str) -> str:
    """
    영역 텍스트가 빈칸일 때, 행 전체에서 키워드로 보조 감지
    """
    t = clean(text)
    if "비문학" in t:
        return "비문학"
    if "문해 독서" in t or "독서" in t:
        # ⚠️ 여기서는 커리큘럼(비문학/문학/문법) 엑셀 기준이라
        # '독서'를 비문학으로 치환하지는 않음.
        pass
    if "문학" in t:
        return "문학"
    if "문법" in t:
        return "문법"
    if "어휘" in t:
        return "어휘"
    return ""


def nonfiction_sentence(content: str) -> str:
    """
    비문학 실전 내용 → 최종 문장
    """
    t = clean(content)
    t = re.sub(r"^\(도서\)\s*", "", t)
    return f"{t} 부분 배경지식 학습과 요약 정리" if t else ""


def grammar_sentence(content: str) -> str:
    """
    문법 개념 내용 → 최종 문장(언어의 본질만 작은 포맷팅)
    """
    t = clean(content)
    if "언어의 본질" in t:
        t2 = re.sub(r"\(\s*\d+\s*\)", "", t).strip()
        tail = re.sub(r"^언어의\s*본질\s*", "", t2).strip()
        return f"언어의 본질 '{tail}'" if tail else "언어의 본질"
    return t


def normalize_lit_concept(content: str) -> str:
    """
    문학 개념/명제판단 등에 붙는 [운문] 같은 태그 제거
    """
    t = clean(content)
    t = re.sub(r"^\[[^\]]+\]\s*", "", t)
    return t


def convert_work_item(item: str) -> str:
    """
    '서시(윤동주)' → '윤동주 <서시>'
    """
    item = clean(item)
    m = re.match(r"(.+?)\((.+?)\)$", item)
    if not m:
        return item
    title = clean(m.group(1))
    author = clean(m.group(2))
    return f"{author} <{title}>"


def normalize_works_list(content: str) -> list[str]:
    """
    '서시(윤동주), 겨울 숲(복효근)' → ['윤동주 <서시>', '복효근 <겨울 숲>']
    """
    t = clean(content)
    if not t:
        return []
    parts = [p.strip() for p in t.split(",") if p.strip()]
    return [convert_work_item(p) for p in parts]


def literature_sentence(concept: str, works: list[str]) -> str:
    """
    문학 최종 문장
    """
    concept = clean(concept)
    works = [clean(w) for w in works if clean(w)]

    if concept and works:
        return f"문학 개념 '{concept}' 공부 후 {', '.join(works)} 작품 분석"
    if concept:
        return f"문학 개념 '{concept}' 공부"
    if works:
        return f"{', '.join(works)} 작품 분석"
    return ""


# -------------------------
# 헤더 감지(시트별 컬럼 위치 추정)
# -------------------------
def detect_columns(df: pd.DataFrame) -> dict | None:
    """
    시트에서 '주차/영역/교재 구성/내용' 헤더가 있는 행을 찾고
    각 컬럼 인덱스를 반환한다.
    """
    for r in range(min(len(df), 60)):
        row = df.iloc[r].tolist()
        row_txt = [clean(x) for x in row]

        def find_exact(val: str):
            return next((i for i, v in enumerate(row_txt) if v == val), None)

        def find_contains(a: str, b: str):
            return next((i for i, v in enumerate(row_txt) if (a in v and b in v)), None)

        week_col = find_exact("주차")
        area_col = find_exact("영역")
        comp_col = find_contains("교재", "구성")
        content_col = find_exact("내용")

        score = sum(x is not None for x in [week_col, area_col, comp_col, content_col])
        if score >= 2:
            # content_col이 없으면 가장 텍스트가 긴 열을 content로 추정
            if content_col is None:
                sample = df.iloc[r + 1 : r + 20]
                best_col, best_len = None, -1
                for c in range(df.shape[1]):
                    ser = sample[c].dropna().astype(str)
                    avg = ser.map(len).mean() if len(ser) else 0
                    if avg > best_len:
                        best_len = avg
                        best_col = c
                content_col = best_col if best_col is not None else 3

            return {
                "header_row": r,
                "week_col": week_col if week_col is not None else 0,
                "area_col": area_col if area_col is not None else 1,
                "comp_col": comp_col if comp_col is not None else 2,
                "content_col": content_col,
            }
    return None


# -------------------------
# 핵심 파싱(최신 안정화)
# -------------------------
def parse_sheet(df: pd.DataFrame, sheet_name: str) -> dict:
    colinfo = detect_columns(df)
    if colinfo is None:
        print(f"\n[시트: {sheet_name}] ❌ 헤더 감지 실패 (주차/영역/교재 구성/내용)")
        return {}

    start_row = colinfo["header_row"] + 1
    week_col = colinfo["week_col"]
    area_col = colinfo["area_col"]
    comp_col = colinfo["comp_col"]
    content_col = colinfo["content_col"]

    issues = {}

    # ✅ 상태값(병합셀 대응)
    current_issue = None
    current_week = None
    current_area = None
    current_comp = None

    # ✅ 헤더 위쪽에 호수가 있을 수도 있어 먼저 스캔(보험)
    for rr in range(start_row):
        issue_no = parse_issue_no(row_join(df.iloc[rr]))
        if issue_no is not None:
            current_issue = issue_no
            issues.setdefault(current_issue, {})
            break

    data = df.iloc[start_row:].copy()

    for _, row in data.iterrows():
        joined = row_join(row)

        # 1) 호수 감지 (행 어디에 있든)
        issue_no = parse_issue_no(joined)
        if issue_no is not None:
            current_issue = issue_no
            issues.setdefault(current_issue, {})
            # 호수 바뀌면 주차/영역 상태 리셋(다음 블록에 영향 방지)
            current_week = None
            current_area = None
            current_comp = None
            continue

        if current_issue is None:
            continue

        # 2) 주차 감지: week_col 우선, 없으면 joined에서 보조 탐색
        wk_txt = clean(row[week_col]) if week_col < len(row) else ""
        week, is_exam = parse_week(wk_txt)

        if week is None:
            week2, is_exam2 = parse_week(joined)
            week = week2
            is_exam = is_exam or is_exam2

        # ✅ 병합셀: 주차가 안 잡히면 이전 주차 유지
        if week is not None:
            current_week = week
        elif current_week is None:
            # 아직 한 번도 주차를 못 잡았으면 이 행은 붙일 곳이 없음
            continue

        # 3) week 객체 준비
        wk_obj = issues[current_issue].setdefault(current_week, {
            "is_exam": False,
            "nonfiction": "",
            "lit_concept": "",
            "lit_works": [],
            "grammar": "",
        })
        wk_obj["is_exam"] |= is_exam

        # 4) 영역 감지: area_col 우선, 비면 joined 보조
        area = clean(row[area_col]) if area_col < len(row) else ""
        if not area:
            area = detect_area_from_text(joined)

        if area in {"비문학", "문학", "문법", "어휘"}:
            current_area = area

        # ✅ 병합셀: 영역이 안 잡히면 이전 영역 유지
        if current_area is None:
            continue

        # 5) 교재 구성(comp): comp_col 우선, 비면 이전 유지
        comp = clean(row[comp_col]) if comp_col < len(row) else ""
        if comp:
            current_comp = comp

        # 아직 comp를 못 잡았으면, joined에서 흔한 키워드로 보조(보험)
        if not current_comp:
            if "실전" in joined:
                current_comp = "실전"
            elif "과제" in joined:
                current_comp = "과제"
            elif "개념" in joined:
                current_comp = "개념"
            elif "명제" in joined:
                current_comp = "명제 판단"

        content = clean(row[content_col]) if content_col < len(row) else ""

        # --------------------
        # 규칙 적용
        # --------------------
        if current_area == "어휘":
            continue  # 어휘 제외

        # 비문학: 실전만
        if current_area == "비문학":
            if current_comp and "실전" in current_comp:
                sent = nonfiction_sentence(content)
                if sent:
                    wk_obj["nonfiction"] = sent
            continue

        # 문학: 개념 + (가능하면 명제판단도 concept 보조로 흡수) + 과제(작품)
        if current_area == "문학":
            if current_comp:
                if "개념" in current_comp or "명제" in current_comp:
                    t = normalize_lit_concept(content)
                    if t:
                        # 개념이 이미 있고 다른 내용이면 이어붙이기(누락 방지)
                        if wk_obj["lit_concept"] and wk_obj["lit_concept"] != t:
                            wk_obj["lit_concept"] = f"{wk_obj['lit_concept']} / {t}"
                        else:
                            wk_obj["lit_concept"] = t
                elif "과제" in current_comp:
                    works = normalize_works_list(content)
                    if works:
                        wk_obj["lit_works"] = works
            continue

        # 문법: 개념만
        if current_area == "문법":
            if current_comp and "개념" in current_comp:
                sent = grammar_sentence(content)
                if sent:
                    wk_obj["grammar"] = sent
            continue

    # ✅ 디버그 로그
    found = sorted(issues.keys())
    print(f"\n[시트: {sheet_name}] header_row={colinfo['header_row']} (week={week_col}, area={area_col}, comp={comp_col}, content={content_col})")
    print("  -> 발견된 호수:", found)
    for ino in found:
        weeks = sorted(list(issues[ino].keys()), key=week_sort_key)
        filled = sum(
            1 for w in issues[ino].values()
            if w.get("nonfiction") or w.get("lit_concept") or w.get("lit_works") or w.get("grammar") or w.get("is_exam")
        )
        print(f"     - {ino}호: 주차 {weeks} / 내용있는 주차수={filled}")

    return issues


def merge_issues(all_issues: list[dict]) -> dict:
    merged = {}
    for issues in all_issues:
        for ino, weeks in issues.items():
            merged.setdefault(ino, {})
            for wk, obj in weeks.items():
                merged[ino].setdefault(wk, {
                    "is_exam": False,
                    "nonfiction": "",
                    "lit_concept": "",
                    "lit_works": [],
                    "grammar": "",
                })
                dst = merged[ino][wk]
                dst["is_exam"] |= obj.get("is_exam", False)
                if obj.get("nonfiction"):
                    dst["nonfiction"] = obj["nonfiction"]
                if obj.get("lit_concept"):
                    dst["lit_concept"] = obj["lit_concept"]
                if obj.get("lit_works"):
                    dst["lit_works"] = obj["lit_works"]
                if obj.get("grammar"):
                    dst["grammar"] = obj["grammar"]
    return merged


# -------------------------
# 렌더링/저장
# -------------------------
def render_issue(issue_no: int, weeks: dict) -> str:
    out = []
    for week in sorted(weeks.keys(), key=week_sort_key):
        w = weeks[week]

        has_any = bool(
            w.get("nonfiction") or w.get("lit_concept") or w.get("lit_works") or w.get("grammar") or w.get("is_exam")
        )
        if not has_any:
            continue

        out += [f"<{week}>", "", ""]

        idx = 1
        if w.get("nonfiction"):
            out += [f"{idx}. 독서(비문학)", w["nonfiction"], "", ""]
            idx += 1

        if w.get("is_exam", False):
            out += [f"{idx}. 정기평가", f"{issue_no}월 정기평가를 시행했습니다.", "", ""]
            idx += 1

        lit = literature_sentence(w.get("lit_concept", ""), w.get("lit_works", []))
        if lit:
            out += [f"{idx}. 문학", lit, "", ""]
            idx += 1

        if w.get("grammar"):
            out += [f"{idx}. 문법", w["grammar"], "", ""]
            idx += 1

    return "\n".join(out).rstrip() + "\n"


def save_txt_files(excel_path: str, output_dir: Path) -> list[Path]:
    xls = pd.ExcelFile(excel_path, engine="openpyxl")

    parsed_list = []
    for sn in xls.sheet_names:
        df = pd.read_excel(excel_path, sheet_name=sn, header=None, engine="openpyxl")
        parsed_list.append(parse_sheet(df, sn))

    issues = merge_issues(parsed_list)

    print("\n[전체 병합 결과] 호수:", sorted(issues.keys()))
    saved = []
    for ino in sorted(issues.keys()):
        text = render_issue(ino, issues[ino])
        out_path = output_dir / f"중1-[{ino}호].txt"
        out_path.write_text(text, encoding="utf-8")
        saved.append(out_path)

    return saved


# -------------------------
# 실행
# -------------------------
saved_files = save_txt_files(EXCEL_PATH, OUTPUT_DIR)

print("\n✅ 저장 완료!")
for p in saved_files:
    print(" -", p)

# ✅ 1호 미리보기(순서 착각 방지: 실제 파일을 찾아봄)
p1 = OUTPUT_DIR / "중1-[1호].txt"
print("\n--- 미리보기: 중1-[1호].txt 상단 120줄 ---")
if p1.exists():
    print("\n".join(p1.read_text(encoding="utf-8").splitlines()[:120]))
else:
    print("❌ 1호 파일이 생성되지 않았습니다. (엑셀에 1호 표기가 없거나, 표기 패턴이 다른 경우)")


[시트: 중1 6~8호] header_row=2 (week=0, area=1, comp=2, content=4)
  -> 발견된 호수: [6, 7, 8]
     - 6호: 주차 ['1주차', '2주차', '3주차', '4주차'] / 내용있는 주차수=4
     - 7호: 주차 ['1주차', '2주차', '3주차', '4주차'] / 내용있는 주차수=4
     - 8호: 주차 ['1주차', '2주차', '3주차', '4주차'] / 내용있는 주차수=4

[전체 병합 결과] 호수: [6, 7, 8]

✅ 저장 완료!
 - C:\Users\9191h\Desktop\[중1] - 1학기 진도 사항\중1-[6호].txt
 - C:\Users\9191h\Desktop\[중1] - 1학기 진도 사항\중1-[7호].txt
 - C:\Users\9191h\Desktop\[중1] - 1학기 진도 사항\중1-[8호].txt

--- 미리보기: 중1-[1호].txt 상단 120줄 ---
<1주차>


1. 독서(비문학)
사피엔스 '서문 ~ 1부 2장' 부분 배경지식 학습과 요약 정리


2. 문학
문학 개념 '시의 화자와 시적 대상 / ① 주체 바꾸기, ② ANY 바꾸기, ⑥ 표현상 특징 파악하기,
 시어·소재(배경)의 의미 및 기능 파악하기' 공부 후 윤동주 <서시>, 복효근 <겨울 숲> 작품 분석


3. 문법
언어의 본질 '기호성, 자의성, 사회성'


<2주차>


1. 독서(비문학)
사피엔스 '1부 3장~ 2부 5장' 부분 배경지식 학습과 요약 정리


2. 문학
문학 개념 '화자의 정서·태도 / ③ 대칭성 바꾸기, ④ 범주 바꾸기, ⑦ 서술상 특징 파악하기,
⑨ 화자·인물의 심리 및 태도 파악하기, ⑩ 시구·구절의 의미 및 기능 파악하기' 공부 후 신석정 <들길에 서서>, 정현종 <떨어져도 튀는 공처럼> 작품 분석


3. 문법
언어의 본질 '역사성, 창조성, 규칙성'


<3주차>


1. 독서(비문학)
사피엔스 '2부 6장~ 8장' 부분 배경지식 학습과 요약 정리

